# defensive_contribution term — P(DC hit) one gameweek ahead

**Render-not-decide.** A re-runnable view of the `defensive_contribution` term, **not** the record — the frozen numbers live in [predictive-phase3-points-model.md](../../../docs/studies/results/predictive-phase3-points-model.md). Logic is in `model/terms/defensive_contribution/`.

A **logistic** term (unlike the Poisson goals/assists/saves): derived binary target `dc_hit = 1{defensive_contribution >= threshold}`, fit **per position** (DEF/MID/FWD; GK exempt). `E[DC points] = DC_POINTS x P(hit)`. See `ASSUMPTIONS.md`.

## Setup

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

from dal.pipeline import load as load_mart
from model.terms.defensive_contribution import DefensiveContributionModel, DefensiveContributionTerm

mart = load_mart()
term = DefensiveContributionTerm(DefensiveContributionModel(variant="selected"))
term.model.pool.minimal, [f.name for f in term.model.pool.candidates]

## Pre-fit assumptions
> Binary term: family is logistic by construction; the floor is class balance (hit rate) + enough hits. See `ASSUMPTIONS.md` §1.

In [ ]:
report = term.model.check_assumptions(DefensiveContributionModel.population(mart))
print(f"detectable={report.detectable}  n_train={report.n_train}")
report.dispersion  # hit_rate / n_hits / family

## Gate — P(hit) vs the lagged DC-action count (dc_roll3)
> Per-term level (spec §5): does the calibrated model out-rank just carrying recent DC form forward? DEF is where DC is most material.

In [ ]:
gate = term.validate(mart)
display(gate.table)

ax = gate.table.set_index("position")[["baseline", "p_dc_hit"]].plot.bar(figsize=(6, 3.5))
ax.set_ylabel("within-position Spearman"); ax.set_title("DC: model vs lagged-count baseline"); plt.tight_layout()

## Diagnose — residuals + ablation
> Post-gate. Worst-missed hits, and the measured contribution of each feature (drop-one, re-gate).

In [ ]:
diag = term.diagnose(mart)
display(diag.ablation)
display(diag.residuals.head(10))